In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import Dict, List
from datetime import datetime

# --- CONFIGURATION AND KEYWORD DICTIONARIES ---

# NOTE: Update the FILE_NAME variable to your exact Scopus CSV file name
FILE_NAME = 'scopus_articles_2017_2026.csv'
ABSTRACT_COLUMN = 'Abstract'
KEYWORDS_COLUMN = 'Author Keywords'

# Standard Stop Words
STOP_WORDS = set(['article', 'study', 'research', 'paper', 'finding', 'aim', 'method', 'show', 'use',
                  'vargo', 'lusch', 'sdl', 'service-dominant', 'this', 'that', 'the', 'a', 'an', 'is',
                  'are', 'and', 'or', 'for', 'of', 'in', 'to', 'as', 'with', 'from', 'we'])

# Q1: Theory Development Score (Weighted Proxy for 0-7 scale)
THEORY_DEVELOPMENT_WEIGHTS = {
    'high_score': ['proposition', 'testable hypothesis', 'causal mechanism', 'theoretical contribution', 'conceptual framework'], # Weight 3
    'mid_score': ['develop a model', 'model', 'framework', 'typology', 'theoretical lens', 'extending theory'], # Weight 2
    'low_score': ['axiom', 'perspective', 'conceptualization', 'implications'] # Weight 1
}

# Q2: Empirical vs. Conceptual Classification
EMPIRICAL_MARKERS = {
    'quantitative': ['regression', 'anova', 'sem', 'structural equation', 'statistical', 'survey data', 'scale development', 'measurement', 'econometric'],
    'qualitative': ['grounded theory', 'thematic analysis', 'narrative', 'interpretive', 'case study', 'interview', 'ethnography', 'coding'],
    'mixed_method': ['mixed method', 'triangulation', 'integrative analysis']
}

# Q3: Comparative Testing
COMPARATIVE_MARKERS = ['comparative', 'comparison', 'versus', 'outperform', 'alternative framework', 'test against', 'cross-country', 'cross-cultural', 'multi-context']

# --- FUNCTIONS FOR ANALYSIS ---

def classify_methodology(text: str, empirical_markers: Dict) -> str:
    """Classifies an article into a methodology type."""
    quant_found = any(re.search(r'\b' + re.escape(k) + r'\b', text) for k in empirical_markers['quantitative'])
    qual_found = any(re.search(r'\b' + re.escape(k) + r'\b', text) for k in empirical_markers['qualitative'])
    mixed_found = any(re.search(r'\b' + re.escape(k) + r'\b', text) for k in empirical_markers['mixed_method'])
    general_empirical = bool(re.search(r'\b(data analysis|evidence|results|finding)\b', text))

    if mixed_found or (quant_found and qual_found):
        return 'Mixed Method'
    elif quant_found:
        return 'Quantitative'
    elif qual_found:
        return 'Qualitative'
    elif general_empirical:
        return 'Empirical (General)'
    else:
        return 'Conceptual/Theoretical'

def score_theory_development(text: str, weights: Dict) -> int:
    """Calculates a weighted score (0-7 proxy) for theoretical rigor."""
    score = 0
    for keyword in weights['high_score']:
        score += len(re.findall(r'\b' + re.escape(keyword) + r'\b', text)) * 3
    for keyword in weights['mid_score']:
        score += len(re.findall(r'\b' + re.escape(keyword) + r'\b', text)) * 2
    for keyword in weights['low_score']:
        score += len(re.findall(r'\b' + re.escape(keyword) + r'\b', text)) * 1
    return min(score, 7) # Cap at 7 to simulate the max scale

def check_comparison(text: str, markers: List) -> bool:
    """Checks for the presence of comparative language (Q3)."""
    return any(re.search(r'\b' + re.escape(k) + r'\b', text) for k in markers)

def preprocess_for_wordcloud(text: str) -> str:
    """Clean text by removing punctuation, numbers, and stop words."""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    meaningful_words = [word for word in words if word not in STOP_WORDS and len(word) > 2]
    return ' '.join(meaningful_words)

# --- EXECUTION WRAPPER ---

def run_sdl_analysis(csv_file: str = FILE_NAME):

    print(f"Loading data from {csv_file}...")
    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"Error: File '{csv_file}' not found. Please upload your Scopus CSV file to Colab.")
        return

    # Clean and prepare columns
    df[ABSTRACT_COLUMN] = df[ABSTRACT_COLUMN].fillna('')
    df[KEYWORDS_COLUMN] = df[KEYWORDS_COLUMN].fillna('')
    # Ensure standard Scopus columns exist for the original module compatibility
    df['Year'] = df['Year'].fillna(df['Year'].median() or 2024).astype(int)
    df['Cited by'] = df['Cited by'].fillna(0).astype(int)

    # Create combined text column
    df['combined_text'] = df[ABSTRACT_COLUMN].str.lower() + ' ' + df[KEYWORDS_COLUMN].str.lower().fillna('')

    # --- APPLY ANALYSIS ---
    print("Applying keyword-weighted content analysis...")
    df['Methodology_Class'] = df['combined_text'].apply(lambda x: classify_methodology(x, EMPIRICAL_MARKERS))
    df['Theory_Development_Score'] = df['combined_text'].apply(lambda x: score_theory_development(x, THEORY_DEVELOPMENT_WEIGHTS))
    df['Has_Comparative_Test'] = df['combined_text'].apply(lambda x: check_comparison(x, COMPARATIVE_MARKERS))

    # --- STANDARDIZATION FOR ACADEMIC OUTPUT ---

    # 1. Rename columns to match the academic module's expected input (XLSX compatibility)
    df.rename(columns={
        'Methodology_Class': 'research_type',
        'Theory_Development_Score': 'theory_dev_score',
        'Has_Comparative_Test': 'is_comparative_study',
    }, inplace=True)

    # 2. Create placeholder/proxy columns required by the sophisticated module's visualization
    # Empirical Score (Proxy: 4 if Empirical, 0 if Conceptual)
    df['empirical_score'] = df['research_type'].apply(lambda x: 4 if x not in ['Conceptual/Theoretical'] else 0)
    # Comparative Score (Proxy: 3 if Comparative, 0 otherwise)
    df['comparative_score'] = df['is_comparative_study'].apply(lambda x: 3 if x else 0)

    # Domain columns (Using simple placeholders, requires manual update for accurate Viz E)
    df['domain_technology'] = df['combined_text'].str.contains(r'\b(ai|digital|platform|technology|system|iot)\b').astype(int)
    df['domain_retail'] = df['combined_text'].str.contains(r'\b(retail|store|shopping|e-commerce|consumer)\b').astype(int)
    df['domain_tourism'] = df['combined_text'].str.contains(r'\b(tourism|travel|hotel|hospitality)\b').astype(int)

    # --- EXPORT ANNOTATED DATA TO XLSX ---
    output_xlsx = 'sdl_annotated_dataset.xlsx'
    df.to_excel(output_xlsx, index=False, engine='openpyxl')
    print(f"\n✅ Data successfully annotated and exported to **{output_xlsx}**.")

    # --- GENERATE VISUALIZATIONS ---
    create_academic_figure(df)

    print("\nAnalysis complete. Check files: **sdl_annotated_dataset.xlsx** and **viz_academic_figure_replicated.png**.")


# --- 2. VISUALIZATION FUNCTION ---

def create_academic_figure(df: pd.DataFrame, output_file: str = 'viz_academic_figure_replicated.png'):
    """
    Creates the publication-ready, multi-panel figure design.
    """
    sns.set_theme(style="whitegrid")

    # --- Setup Figure and Grid ---
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
    N = len(df)

    fig.suptitle(f'SDL Post-2017 Empirical Validation Assessment\nSystematic Analysis of {N} Articles (2017-{df["Year"].max()})',
                 fontsize=18, fontweight='bold', y=0.98)

    # --- Panel A: Research Type Distribution Over Time (Area Chart) ---
    ax1 = fig.add_subplot(gs[0, :2])
    research_by_year = pd.crosstab(df['Year'], df['research_type'], normalize='index') * 100
    expected_types = ['Conceptual/Theoretical', 'Quantitative', 'Qualitative', 'Mixed Method', 'Empirical (General)']
    for t in expected_types:
        if t not in research_by_year.columns:
            research_by_year[t] = 0

    colors = {'Conceptual/Theoretical': '#e74c3c', 'Quantitative': '#3498db',
              'Qualitative': '#2ecc71', 'Mixed Method': '#95a5a6', 'Empirical (General)': '#f1c40f'}

    research_by_year[expected_types].plot(kind='area', stacked=True, ax=ax1,
                                          color=[colors[t] for t in expected_types], alpha=0.8, legend=False)

    ax1.set_title('A. Research Methodology Evolution', fontsize=14, fontweight='bold', loc='left')
    ax1.set_xlabel('Year', fontsize=12)
    ax1.set_ylabel('Percentage of Articles (%)', fontsize=12)
    ax1.legend(title='Research Type', loc='upper left', fontsize=10)
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim(0, 100)

    # --- Panel B: Average Scores (Bar Chart) ---
    ax2 = fig.add_subplot(gs[0, 2])
    scores_df = pd.DataFrame({
        'Theory\nDev (0-7)': [df['theory_dev_score'].mean()],
        'Empirical\nScore (0-8)': [df['empirical_score'].mean()],
        'Comparative\nTest (0-5)': [df['comparative_score'].mean()]
    })

    scores_df.T.plot(kind='barh', ax=ax2, legend=False, color='#34495e', alpha=0.8)
    ax2.set_title('B. Average Rigor Scores', fontsize=14, fontweight='bold', loc='left')
    ax2.set_xlabel('Mean Score', fontsize=12)
    ax2.set_yticks(ticks=range(len(scores_df.columns)), labels=scores_df.columns)
    ax2.tick_params(axis='y', rotation=0)
    ax2.set_xlim(0, 7.5)

    # --- Panel C: Empirical vs Conceptual Trend (Line Chart) ---
    ax3 = fig.add_subplot(gs[1, :2])
    empirical_trend = df.groupby('Year').apply(
        lambda x: (x['research_type'] != 'Conceptual/Theoretical').sum() / len(x) * 100
    )

    ax3.plot(empirical_trend.index, empirical_trend.values, marker='o', linewidth=3,
             markersize=8, color='#27ae60', label='Empirical')
    ax3.plot(empirical_trend.index, 100-empirical_trend.values, marker='s', linewidth=3,
             markersize=8, color='#e74c3c', label='Conceptual')

    ax3.set_title('C. Empirical vs Conceptual Research Trend', fontsize=14, fontweight='bold', loc='left')
    ax3.set_xlabel('Year', fontsize=12)
    ax3.set_ylabel('Percentage (%)', fontsize=12)
    ax3.legend(fontsize=10)
    ax3.grid(axis='y', alpha=0.3)
    ax3.set_ylim(0, 100)
    ax3.axhline(y=50, color='gray', linestyle='--', alpha=0.6, label='50% threshold')

    # --- Panel D: Comparative Testing Rate (Bar Chart) ---
    ax4 = fig.add_subplot(gs[1, 2])
    comp_rate = (df['is_comparative_study'].sum() / N) * 100
    color = '#e74c3c' if comp_rate < 5 else '#f39c12' if comp_rate < 10 else '#27ae60'

    ax4.bar(['Comparative\nStudies'], [comp_rate], color=color, alpha=0.8)
    ax4.set_title('D. Comparative Testing Rate', fontsize=14, fontweight='bold', loc='left')
    ax4.set_ylabel('Percentage (%)', fontsize=12)
    ax4.set_ylim(0, 15)
    ax4.axhline(y=5, color='red', linestyle='--', alpha=0.6, linewidth=2, label='Critical Threshold')
    ax4.text(0, comp_rate + 0.5, f'{comp_rate:.1f}%', ha='center', fontweight='bold', fontsize=12)
    ax4.tick_params(axis='x', rotation=0)

    # --- Panel E: Domain Distribution ---
    ax5 = fig.add_subplot(gs[2, :])
    domain_cols = [col for col in df.columns if col.startswith('domain_')]
    if domain_cols:
        domain_data = df[domain_cols].sum().sort_values(ascending=False)
        domain_data.index = [idx.replace('domain_', '').title() for idx in domain_data.index]
        domain_data.plot(kind='bar', ax=ax5, color='#3498db', alpha=0.7)
    else:
        # Fallback if domain detection fails
        ax5.bar(['N/A'], [0])

    ax5.set_title('E. SDL Application Domain Distribution', fontsize=14, fontweight='bold', loc='left')
    ax5.set_xlabel('Domain', fontsize=12)
    ax5.set_ylabel('Number of Articles', fontsize=12)
    ax5.set_xticklabels(ax5.get_xticklabels(), rotation=45, ha='right')
    ax5.grid(axis='y', alpha=0.3)

    # --- Final Polish ---
    note_text = (f"Note: Analysis based on {N} articles. | Empirical rate: {(df['research_type'] != 'Conceptual/Theoretical').sum()/N*100:.1f}% | "
                 f"Comparative studies: {df['is_comparative_study'].sum()/N*100:.1f}% | Theory development (score\u22653): {(df['theory_dev_score'] >= 3).sum()/N*100:.1f}%")
    fig.text(0.5, 0.01, note_text, ha='center', fontsize=10, style='italic', color='gray')

    plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

# --- RUN SCRIPT IN COLAB ---
# This is the single function call you execute in Colab after uploading your CSV.

run_sdl_analysis(FILE_NAME)

Loading data from scopus_articles_2017_2026.csv...
Applying keyword-weighted content analysis...


/tmp/ipython-input-1827808218.py:126: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['domain_technology'] = df['combined_text'].str.contains(r'\b(ai|digital|platform|technology|system|iot)\b').astype(int)
/tmp/ipython-input-1827808218.py:127: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['domain_retail'] = df['combined_text'].str.contains(r'\b(retail|store|shopping|e-commerce|consumer)\b').astype(int)
/tmp/ipython-input-1827808218.py:128: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['domain_tourism'] = df['combined_text'].str.contains(r'\b(tourism|travel|hotel|hospitality)\b').astype(int)



✅ Data successfully annotated and exported to **sdl_annotated_dataset.xlsx**.


/tmp/ipython-input-1827808218.py:195: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  empirical_trend = df.groupby('Year').apply(



Analysis complete. Check files: **sdl_annotated_dataset.xlsx** and **viz_academic_figure_replicated.png**.


In [ ]:
from google.colab import files
files.download('sdl_annotated_dataset.xlsx')

FileNotFoundError: Cannot find file: sdl_annotated_dataset.xlsx

In [ ]:
from google.colab import files
files.download('sdl_annotated_dataset.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>